# 03 Rules Engine — SENTINEL
Output for 04: `*_fe.csv` enriched with rule cols. Recommend `rule_score_weighted >= 3/4` (means 1.1 normal vs 3.9 suspicious).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve().parent))


In [ ]:
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support
from src.rules_engine import apply_rules
from src import config


In [ ]:
train = pd.read_csv(config.ARTIFACTS_DIR / 'train_fe.csv')
valid = pd.read_csv(config.ARTIFACTS_DIR / 'valid_fe.csv')
test = pd.read_csv(config.ARTIFACTS_DIR / 'test_fe.csv')
train = apply_rules(train)
valid = apply_rules(valid)
test = apply_rules(test)
print(train[['rule_score','rule_score_weighted']].describe())
print(train.groupby('is_suspicious')[['rule_score','rule_score_weighted']].mean())


In [ ]:
# Flag precision/recall vs is_suspicious; mean rule_score_weighted ~1.1 (normal) vs ~3.9 (suspicious)
flags = [c for c in train.columns if c.endswith('_flag') or c in ('new_beneficiary_risk',)]
for f in flags:
    p, r, _, _ = precision_recall_fscore_support(train['is_suspicious'], train[f], average='binary', zero_division=0)
    print(f'{f:28s} prec={p:.3f} rec={r:.3f} sup={int(train[f].sum())}')
for t in [3, 4]:
    pred = (train['rule_score_weighted'] >= t).astype(int)
    p, r, _, _ = precision_recall_fscore_support(train['is_suspicious'], pred, average='binary', zero_division=0)
    print(f'weighted>={t} prec={p:.3f} rec={r:.3f}')
print('RECOMMEND: rule_score_weighted >= 3 (recall) / >= 4 (precision)')


In [ ]:
train.to_csv(config.ARTIFACTS_DIR / 'train_fe.csv', index=False)
valid.to_csv(config.ARTIFACTS_DIR / 'valid_fe.csv', index=False)
test.to_csv(config.ARTIFACTS_DIR / 'test_fe.csv', index=False)
print('saved *_fe.csv with rule cols -> used by 04_isolation_forest.ipynb')
